In [3]:
!pip install tensorflow

In [ ]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import numpy as np
import scipy.sparse as sp
from scipy.sparse import linalg as sp_linalg
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()   # 以 TF1 静态图模式运行

from tensorflow.compat.v1.nn.rnn_cell import RNNCell   # 替代 tf.contrib.rnn.RNNCell


# ── 图工具函数（内联替代 lib/utils）──────────────────────────────────────────

def calculate_random_walk_matrix(adj_mx):
    """D^{-1} A，行归一化随机游走矩阵."""
    adj_mx = sp.coo_matrix(adj_mx)
    d = np.array(adj_mx.sum(1))
    d_inv = np.power(d, -1).flatten()
    d_inv[np.isinf(d_inv)] = 0.
    return sp.diags(d_inv).dot(adj_mx).tocoo()


def _calc_normalized_laplacian(adj):
    adj = sp.coo_matrix(adj)
    d = np.array(adj.sum(1))
    d_inv_sqrt = np.power(d, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    D = sp.diags(d_inv_sqrt)
    # L = I - D^{-1/2} A D^{-1/2}
    return sp.eye(adj.shape[0]) - D.dot(adj).dot(D).tocoo()


def calculate_scaled_laplacian(adj_mx, lambda_max=None, undirected=True):
    """对称归一化拉普拉斯，缩放到 [-1, 1]."""
    if undirected:
        adj_mx = np.maximum(adj_mx, adj_mx.T)
    L = _calc_normalized_laplacian(adj_mx)
    if lambda_max is None:
        lambda_max, _ = sp_linalg.eigsh(L, 1, which='LM')
        lambda_max = lambda_max[0]
    L = sp.csr_matrix(L)
    I = sp.eye(L.shape[0])
    L = (2 / lambda_max * L) - I
    return L.astype(np.float32).tocoo()


# ── DCGRUCell ─────────────────────────────────────────────────────────────────

class DCGRUCell(RNNCell):
    """Diffusion Convolutional GRU Cell（扩散卷积门控循环单元）.

    将图上的扩散过程（Diffusion Process）嵌入 GRU 的更新机制中，
    替换标准 GRU 中的全连接矩阵乘法，从而捕捉空间依赖。
    """

    def call(self, inputs, **kwargs):
        pass

    def compute_output_shape(self, input_shape):
        pass

    def __init__(self, num_units, adj_mx, max_diffusion_step, num_nodes, num_proj=None,
                 activation=tf.nn.tanh, reuse=None, filter_type="laplacian", use_gc_for_ru=True):
        super(DCGRUCell, self).__init__(_reuse=reuse)
        self._activation = activation
        self._num_nodes = num_nodes
        self._num_proj = num_proj
        self._num_units = num_units
        self._max_diffusion_step = max_diffusion_step
        self._supports = []
        self._use_gc_for_ru = use_gc_for_ru

        supports = []
        if filter_type == "laplacian":
            supports.append(calculate_scaled_laplacian(adj_mx, lambda_max=None))
        elif filter_type == "random_walk":
            supports.append(calculate_random_walk_matrix(adj_mx).T)
        elif filter_type == "dual_random_walk":
            supports.append(calculate_random_walk_matrix(adj_mx).T)
            supports.append(calculate_random_walk_matrix(adj_mx.T).T)
        else:
            supports.append(calculate_scaled_laplacian(adj_mx))

        for support in supports:
            self._supports.append(self._build_sparse_matrix(support))

    @staticmethod
    def _build_sparse_matrix(L):
        """scipy sparse → tf.SparseTensor（COO 格式）."""
        L = L.tocoo()
        indices = np.column_stack((L.row, L.col))
        L = tf.SparseTensor(indices, L.data.astype(np.float32), L.shape)
        return tf.sparse.reorder(L)   # TF2 中 tf.sparse_reorder 改名为此

    @property
    def state_size(self):
        return self._num_nodes * self._num_units

    @property
    def output_size(self):
        output_size = self._num_nodes * self._num_units
        if self._num_proj is not None:
            output_size = self._num_nodes * self._num_proj
        return output_size

    def __call__(self, inputs, state, scope=None):
        with tf.variable_scope(scope or "dcgru_cell"):
            with tf.variable_scope("gates"):
                output_size = 2 * self._num_units
                fn = self._gconv if self._use_gc_for_ru else self._fc
                value = tf.nn.sigmoid(fn(inputs, state, output_size, bias_start=1.0))
                value = tf.reshape(value, (-1, self._num_nodes, output_size))
                r, u = tf.split(value=value, num_or_size_splits=2, axis=-1)
                r = tf.reshape(r, (-1, self._num_nodes * self._num_units))
                u = tf.reshape(u, (-1, self._num_nodes * self._num_units))

            with tf.variable_scope("candidate"):
                c = self._gconv(inputs, r * state, self._num_units)
                if self._activation is not None:
                    c = self._activation(c)

            output = new_state = u * state + (1 - u) * c

            if self._num_proj is not None:
                with tf.variable_scope("projection"):
                    w = tf.get_variable('w', shape=(self._num_units, self._num_proj))
                    batch_size = inputs.get_shape()[0].value
                    output = tf.reshape(new_state, shape=(-1, self._num_units))
                    output = tf.reshape(tf.matmul(output, w), shape=(batch_size, self.output_size))

        return output, new_state

    @staticmethod
    def _concat(x, x_):
        x_ = tf.expand_dims(x_, 0)
        return tf.concat([x, x_], axis=0)

    def _fc(self, inputs, state, output_size, bias_start=0.0):
        dtype = inputs.dtype
        batch_size = inputs.get_shape()[0].value
        inputs = tf.reshape(inputs, (batch_size * self._num_nodes, -1))
        state = tf.reshape(state, (batch_size * self._num_nodes, -1))
        inputs_and_state = tf.concat([inputs, state], axis=-1)
        input_size = inputs_and_state.get_shape()[-1].value
        weights = tf.get_variable(
            'weights', [input_size, output_size], dtype=dtype,
            initializer=tf.glorot_uniform_initializer())   # xavier = glorot_uniform，TF2 写法
        value = tf.nn.sigmoid(tf.matmul(inputs_and_state, weights))
        biases = tf.get_variable("biases", [output_size], dtype=dtype,
                                 initializer=tf.constant_initializer(bias_start, dtype=dtype))
        return tf.nn.bias_add(value, biases)

    def _gconv(self, inputs, state, output_size, bias_start=0.0):
        """K 阶扩散卷积：切比雪夫递推 x^(k) = 2S·x^(k-1) - x^(k-2)."""
        batch_size = inputs.get_shape()[0].value
        inputs = tf.reshape(inputs, (batch_size, self._num_nodes, -1))
        state  = tf.reshape(state,  (batch_size, self._num_nodes, -1))
        inputs_and_state = tf.concat([inputs, state], axis=2)   # (B, N, D)
        input_size = inputs_and_state.get_shape()[2].value
        dtype = inputs.dtype

        x0 = tf.transpose(inputs_and_state, perm=[1, 2, 0])    # (N, D, B)
        x0 = tf.reshape(x0, [self._num_nodes, input_size * batch_size])
        x  = tf.expand_dims(x0, axis=0)                        # (1, N, D*B)

        scope = tf.get_variable_scope()
        with tf.variable_scope(scope):
            if self._max_diffusion_step > 0:
                for support in self._supports:
                    # tf.sparse_tensor_dense_matmul 在 TF2 中改名为 sparse.sparse_dense_matmul
                    x1 = tf.sparse.sparse_dense_matmul(support, x0)
                    x  = self._concat(x, x1)
                    for k in range(2, self._max_diffusion_step + 1):
                        x2 = 2 * tf.sparse.sparse_dense_matmul(support, x1) - x0
                        x  = self._concat(x, x2)
                        x1, x0 = x2, x1

            num_matrices = len(self._supports) * self._max_diffusion_step + 1
            x = tf.reshape(x, [num_matrices, self._num_nodes, input_size, batch_size])
            x = tf.transpose(x, perm=[3, 1, 2, 0])             # (B, N, D, num_matrices)
            x = tf.reshape(x, [batch_size * self._num_nodes, input_size * num_matrices])

            weights = tf.get_variable(
                'weights', [input_size * num_matrices, output_size], dtype=dtype,
                initializer=tf.glorot_uniform_initializer())
            x = tf.matmul(x, weights)
            biases = tf.get_variable("biases", [output_size], dtype=dtype,
                                     initializer=tf.constant_initializer(bias_start, dtype=dtype))
            x = tf.nn.bias_add(x, biases)

        return tf.reshape(x, [batch_size, self._num_nodes * output_size])


In [7]:
!pip install tensorflow-addons

ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons


In [ ]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

# tf 已在上一个 cell 中通过 tensorflow.compat.v1 导入，这里直接使用
import numpy as np


# ── legacy_seq2seq.rnn_decoder 手动实现（tf.contrib 在 TF2 中被移除）────────
def rnn_decoder(decoder_inputs, initial_state, cell, loop_function=None, scope=None):
    """逐步展开解码器 RNN，权重在所有时间步之间共享.

    :param decoder_inputs:  list of (B, input_size) tensors，长度 = H+1（含 GO token）
    :param initial_state:   编码器最终隐状态（MultiRNNCell 时为 tuple）
    :param cell:            RNN cell（MultiRNNCell）
    :param loop_function:   fn(prev_output, step_i) → next_input；None 则直接用 decoder_inputs[i]
    :return: (outputs, final_state)
    """
    with tf.variable_scope(scope or "rnn_decoder"):
        state = initial_state
        outputs = []
        prev = None
        for i, inp in enumerate(decoder_inputs):
            if loop_function is not None and prev is not None:
                inp = loop_function(prev, i)   # 训练时可能用真值，测试时用上步预测
            if i > 0:
                tf.get_variable_scope().reuse_variables()   # 所有步共享同一套 RNN 权重
            output, state = cell(inp, state)
            outputs.append(output)
            if loop_function is not None:
                prev = output
        return outputs, state


# ── 带掩码的 MSE 损失 ────────────────────────────────────────────────────────
def masked_mse_tf(preds, labels, null_val=np.nan):
    """忽略缺失值（null_val 位置）的 MSE 损失."""
    if np.isnan(null_val):
        mask = ~tf.math.is_nan(labels)   # tf.is_nan 在 TF2 中移至 tf.math.is_nan
    else:
        mask = tf.not_equal(labels, null_val)
    mask = tf.cast(mask, tf.float32)
    mask /= tf.reduce_mean(mask)
    mask = tf.where(tf.math.is_nan(mask), tf.zeros_like(mask), mask)
    loss = tf.square(tf.subtract(preds, labels))
    loss = loss * mask
    loss = tf.where(tf.math.is_nan(loss), tf.zeros_like(loss), loss)
    return tf.reduce_mean(loss)


# ── DCRNN 模型 ────────────────────────────────────────────────────────────────
class DCRNNModel(object):
    """DCRNN 完整模型：编码器-解码器架构，解码器支持 Scheduled Sampling."""

    def __init__(self, is_training, batch_size, scaler, adj_mx, **model_kwargs):
        self._scaler  = scaler
        self._loss    = None
        self._mae     = None
        self._train_op = None

        max_diffusion_step       = int(model_kwargs.get('max_diffusion_step', 2))
        cl_decay_steps           = int(model_kwargs.get('cl_decay_steps', 1000))
        filter_type              = model_kwargs.get('filter_type', 'laplacian')
        horizon                  = int(model_kwargs.get('horizon', 1))
        num_nodes                = int(model_kwargs.get('num_nodes', 1))
        num_rnn_layers           = int(model_kwargs.get('num_rnn_layers', 1))
        rnn_units                = int(model_kwargs.get('rnn_units'))
        seq_len                  = int(model_kwargs.get('seq_len'))
        use_curriculum_learning  = bool(model_kwargs.get('use_curriculum_learning', False))
        input_dim                = int(model_kwargs.get('input_dim', 1))
        output_dim               = int(model_kwargs.get('output_dim', 1))

        # 占位符：TF2 compat.v1 下 tf.placeholder 仍可用
        self._inputs = tf.placeholder(tf.float32,
                                      shape=(batch_size, seq_len, num_nodes, input_dim),
                                      name='inputs')
        self._labels = tf.placeholder(tf.float32,
                                      shape=(batch_size, horizon, num_nodes, input_dim),
                                      name='labels')

        GO_SYMBOL = tf.zeros(shape=(batch_size, num_nodes * output_dim))

        cell = DCGRUCell(rnn_units, adj_mx, max_diffusion_step=max_diffusion_step,
                         num_nodes=num_nodes, filter_type=filter_type)
        cell_with_proj = DCGRUCell(rnn_units, adj_mx, max_diffusion_step=max_diffusion_step,
                                   num_nodes=num_nodes, num_proj=output_dim, filter_type=filter_type)

        # tf.contrib.rnn.MultiRNNCell → tf.compat.v1.nn.rnn_cell.MultiRNNCell
        encoding_cells = tf.compat.v1.nn.rnn_cell.MultiRNNCell(
            [cell] * num_rnn_layers, state_is_tuple=True)
        decoding_cells = tf.compat.v1.nn.rnn_cell.MultiRNNCell(
            [cell] * (num_rnn_layers - 1) + [cell_with_proj], state_is_tuple=True)

        global_step = tf.train.get_or_create_global_step()

        with tf.variable_scope('DCRNN_SEQ'):
            inputs = tf.unstack(
                tf.reshape(self._inputs, (batch_size, seq_len, num_nodes * input_dim)), axis=1)
            labels = tf.unstack(
                tf.reshape(self._labels[..., :output_dim],
                           (batch_size, horizon, num_nodes * output_dim)), axis=1)
            labels.insert(0, GO_SYMBOL)   # [GO, y_1, ..., y_H]

            def _loop_function(prev, i):
                if is_training:
                    if use_curriculum_learning:
                        # tf.random_uniform → tf.random.uniform（TF2 写法）
                        c = tf.random.uniform((), minval=0, maxval=1.)
                        threshold = self._compute_sampling_threshold(global_step, cl_decay_steps)
                        return tf.cond(tf.less(c, threshold), lambda: labels[i], lambda: prev)
                    else:
                        return labels[i]
                else:
                    return prev   # 测试时完全自回归

            # tf.contrib.rnn.static_rnn → tf.compat.v1.nn.static_rnn
            _, enc_state = tf.compat.v1.nn.static_rnn(encoding_cells, inputs, dtype=tf.float32)
            # legacy_seq2seq.rnn_decoder → 上面手动实现的 rnn_decoder
            outputs, _ = rnn_decoder(labels, enc_state, decoding_cells,
                                     loop_function=_loop_function)

        outputs = tf.stack(outputs[:-1], axis=1)   # 丢弃对 GO token 的响应
        self._outputs = tf.reshape(outputs, (batch_size, horizon, num_nodes, output_dim),
                                   name='outputs')
        self._merged = tf.compat.v1.summary.merge_all()   # tf.summary.merge_all 的 compat 写法

    @staticmethod
    def _compute_sampling_threshold(global_step, k):
        """逆 Sigmoid 衰减：threshold(t) = k / (k + exp(t/k))，从 1 衰减到 0."""
        return tf.cast(k / (k + tf.exp(global_step / k)), tf.float32)

    @property
    def inputs(self):
        return self._inputs

    @property
    def labels(self):
        return self._labels

    @property
    def loss(self):
        return self._loss

    @property
    def mae(self):
        return self._mae

    @property
    def merged(self):
        return self._merged

    @property
    def outputs(self):
        return self._outputs


In [ ]:
import numpy as np
import os

# ── 归一化器 ──────────────────────────────────────────────────────────────────
class StandardScaler:
    def __init__(self, mean, std):
        self.mean = mean
        self.std  = std

    def transform(self, data):
        return (data - self.mean) / self.std

    def inverse_transform(self, data):
        return (data * self.std) + self.mean


# ── 纯 numpy 批迭代器（替代 PyTorch DataLoader）──────────────────────────────
class NumpyLoader:
    """按 batch_size 切分 numpy 数组并迭代，不足一个 batch 的尾批直接丢弃.

    TF1 静态图要求 placeholder 形状固定，因此必须丢弃尾批。
    """
    def __init__(self, x, y, batch_size, shuffle=False):
        self.x          = x
        self.y          = y
        self.batch_size = batch_size
        self.shuffle    = shuffle
        # 丢弃尾批后的完整 batch 数量
        self.n_batches  = len(x) // batch_size

    def __len__(self):
        return self.n_batches

    def __iter__(self):
        idx = np.arange(len(self.x))
        if self.shuffle:
            np.random.shuffle(idx)
        for i in range(self.n_batches):
            sel = idx[i * self.batch_size : (i + 1) * self.batch_size]
            yield self.x[sel], self.y[sel]


# ── 数据集加载 ─────────────────────────────────────────────────────────────────
def load_dataset(dataset_dir, batch_size, valid_batch_size=None, test_batch_size=None):
    """加载 npz 数据集，归一化后返回 NumpyLoader 字典.

    返回字典键：train_loader / val_loader / test_loader / scaler
    x shape: (samples, N, P, C)  →  DCRNN 训练时会在 to_dcrnn_input 中转置
    y shape: (samples, N, H, C)
    """
    valid_batch_size = valid_batch_size or batch_size
    test_batch_size  = test_batch_size  or batch_size

    data = {}
    for category in ["train", "val", "test"]:
        cat_data = np.load(os.path.join(dataset_dir, category + ".npz"))
        data["x_" + category] = cat_data["x"].astype(np.float32)
        data["y_" + category] = cat_data["y"].astype(np.float32)

    # Z-score 归一化（只对第 0 通道，即流量）
    scaler = StandardScaler(
        mean=data["x_train"][..., 0].mean(),
        std =data["x_train"][..., 0].std()
    )
    for category in ["train", "val", "test"]:
        data["x_" + category][..., 0] = scaler.transform(data["x_" + category][..., 0])

    data["train_loader"] = NumpyLoader(data["x_train"], data["y_train"],
                                       batch_size=batch_size,       shuffle=True)
    data["val_loader"]   = NumpyLoader(data["x_val"],   data["y_val"],
                                       batch_size=valid_batch_size, shuffle=False)
    data["test_loader"]  = NumpyLoader(data["x_test"],  data["y_test"],
                                       batch_size=test_batch_size,  shuffle=False)
    data["scaler"] = scaler
    return data


In [ ]:
# ── DCRNN 训练框架 ────────────────────────────────────────────────────────────
# tf 已在 cell 1 以 tensorflow.compat.v1 导入并 disable_v2_behavior，此处直接用
import numpy as np
import time
import os
import logging
from tqdm import tqdm

tf.set_random_seed(313)   # compat.v1 下 tf.set_random_seed 仍可用
np.random.seed(313)

# ── 超参数 ────────────────────────────────────────────────────────────────────
batch_size         = 64
lrate              = 5e-4
wdecay             = 5e-5
epochs             = 300
num_nodes          = 266
input_len          = 12
output_len         = 12
input_dim          = 1
output_dim         = 1
rnn_units          = 64
num_rnn_layers     = 2
max_diffusion_step = 2
filter_type        = "dual_random_walk"
cl_decay_steps     = 2000
max_grad_norm      = 5.0
dataset_dir        = "w_texi_drop"
SAVE_DIR           = "./checkpoints"
SAVE_AFTER         = 40
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Logger ────────────────────────────────────────────────────────────────────
def get_logger(log_path):
    logger = logging.getLogger("dcrnn")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s  %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    fh = logging.FileHandler(log_path, mode="a", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger

# ── 指标计算 ──────────────────────────────────────────────────────────────────
def calc_metrics(pred, real):
    mae  = np.mean(np.abs(pred - real))
    rmse = np.sqrt(np.mean((pred - real) ** 2))
    mask = real != 0
    mape = np.mean(np.abs((pred[mask] - real[mask]) / real[mask])) * 100
    wape = np.sum(np.abs(pred - real)) / np.sum(np.abs(real)) * 100
    return {"mae": mae, "rmse": rmse, "mape": mape, "wape": wape}

# ── 数据格式转换 ──────────────────────────────────────────────────────────────
def to_dcrnn_input(x_batch, y_batch):
    """NumpyLoader 输出 (B,N,P,C)/(B,N,H,C) → DCRNN 需要的 (B,P,N,1)/(B,H,N,1)."""
    x = x_batch[..., 0:1].transpose(0, 2, 1, 3)
    y = y_batch[..., 0:1].transpose(0, 2, 1, 3)
    return x, y

# ── 构建计算图 ─────────────────────────────────────────────────────────────────
def build_graph(adj_mx):
    model_kwargs = {
        "max_diffusion_step"    : max_diffusion_step,
        "cl_decay_steps"        : cl_decay_steps,
        "filter_type"           : filter_type,
        "horizon"               : output_len,
        "max_grad_norm"         : max_grad_norm,
        "num_nodes"             : num_nodes,
        "num_rnn_layers"        : num_rnn_layers,
        "rnn_units"             : rnn_units,
        "seq_len"               : input_len,
        "use_curriculum_learning": True,
        "input_dim"             : input_dim,
        "output_dim"            : output_dim,
    }
    with tf.variable_scope("DCRNN") as scope:
        train_model = DCRNNModel(is_training=True,  batch_size=batch_size,
                                 scaler=None, adj_mx=adj_mx, **model_kwargs)
        scope.reuse_variables()
        eval_model  = DCRNNModel(is_training=False, batch_size=batch_size,
                                 scaler=None, adj_mx=adj_mx, **model_kwargs)

    train_loss = masked_mse_tf(train_model.outputs, train_model.labels[..., :output_dim])

    l2_vars    = [v for v in tf.trainable_variables() if "bias" not in v.name.lower()]
    l2_reg     = tf.add_n([tf.nn.l2_loss(v) for v in l2_vars]) * wdecay
    total_loss = train_loss + l2_reg

    global_step = tf.train.get_or_create_global_step()
    optimizer   = tf.train.AdamOptimizer(learning_rate=lrate)   # compat.v1 下 tf.train 仍可用
    grads, vars_ = zip(*optimizer.compute_gradients(total_loss))
    grads, _     = tf.clip_by_global_norm(grads, max_grad_norm)
    train_op     = optimizer.apply_gradients(zip(grads, vars_), global_step=global_step)

    return train_model, eval_model, train_loss, train_op

# ── 评估 ──────────────────────────────────────────────────────────────────────
def evaluate(sess, eval_model, loader, scaler):
    preds, reals, mse_list = [], [], []
    for x_batch, y_batch in loader:
        x_np, y_np = to_dcrnn_input(x_batch, y_batch)
        if x_np.shape[0] != batch_size:
            continue
        feed = {eval_model.inputs: x_np, eval_model.labels: y_np}
        pred_scaled = sess.run(eval_model.outputs, feed_dict=feed)   # (B, H, N, 1)

        pred = scaler.inverse_transform(pred_scaled.reshape(-1)).reshape(pred_scaled.shape[:-1])
        real = scaler.inverse_transform(y_np[..., 0].reshape(-1)).reshape(y_np[..., 0].shape)

        mse_list.append(np.mean((pred - real) ** 2))
        preds.append(pred)
        reals.append(real)

    pred_all = np.concatenate(preds, axis=0)
    real_all = np.concatenate(reals, axis=0)
    metrics  = calc_metrics(pred_all, real_all)
    metrics["mse"] = np.mean(mse_list)
    return metrics

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    log_path  = os.path.join(SAVE_DIR, "dcrnn_train.log")
    ckpt_path = os.path.join(SAVE_DIR, "dcrnn_best_model")
    logger    = get_logger(log_path)

    dataloader = load_dataset(dataset_dir, batch_size=batch_size,
                              valid_batch_size=batch_size, test_batch_size=batch_size)
    scaler = dataloader["scaler"]
    adj_mx = np.load(os.path.join(dataset_dir, "adj_mx.npy")).astype(np.float32)

    train_model, eval_model, train_loss_op, train_op = build_graph(adj_mx)
    saver = tf.train.Saver(max_to_keep=1)   # compat.v1 下 tf.train.Saver 仍可用

    gpu_cfg = tf.ConfigProto()             # compat.v1 下 tf.ConfigProto 仍可用
    gpu_cfg.gpu_options.allow_growth = True

    best_val_mae = float("inf")
    logger.info(f"rnn_units={rnn_units}  layers={num_rnn_layers}  K={max_diffusion_step}  filter={filter_type}")
    logger.info(f"Training started  |  epochs={epochs}  lr={lrate}  wd={wdecay}")
    logger.info(f"Checkpoint dir: {SAVE_DIR}  |  save_after={SAVE_AFTER}")
    logger.info(f"{'Epoch':>6}  {'TrainMSE':>10}  {'MAE':>8}  {'RMSE':>8}  {'MAPE%':>8}  {'WAPE%':>8}  {'Time':>7}  Note")

    with tf.Session(config=gpu_cfg) as sess:   # compat.v1 下 tf.Session 仍可用
        sess.run(tf.global_variables_initializer())

        for epoch in range(1, epochs + 1):
            t0 = time.time()

            train_losses = []
            train_bar = tqdm(dataloader["train_loader"],
                             desc=f"Epoch {epoch:>3}/{epochs} [train]",
                             leave=False, dynamic_ncols=True)
            for x_batch, y_batch in train_bar:
                x_np, y_np = to_dcrnn_input(x_batch, y_batch)
                if x_np.shape[0] != batch_size:
                    continue
                feed = {train_model.inputs: x_np, train_model.labels: y_np}
                loss_val, _ = sess.run([train_loss_op, train_op], feed_dict=feed)
                train_losses.append(loss_val)
                train_bar.set_postfix(loss=f"{loss_val:.4f}")

            val_m      = evaluate(sess, eval_model, dataloader["val_loader"], scaler)
            epoch_time = time.time() - t0

            tag = ""
            if epoch > SAVE_AFTER and val_m["mae"] < best_val_mae:
                best_val_mae = val_m["mae"]
                saver.save(sess, ckpt_path)
                tag = "★ best"

            logger.info(
                f"{epoch:>6}/{epochs}  "
                f"{np.mean(train_losses):>10.4f}  "
                f"{val_m['mae']:>8.4f}  "
                f"{val_m['rmse']:>8.4f}  "
                f"{val_m['mape']:>8.2f}  "
                f"{val_m['wape']:>8.2f}  "
                f"{epoch_time:>6.1f}s  {tag}"
            )

        logger.info("=" * 70)
        logger.info(f"Loading best checkpoint  (best val MAE={best_val_mae:.4f})")
        saver.restore(sess, ckpt_path)

        test_m = evaluate(sess, eval_model, dataloader["test_loader"], scaler)
        logger.info(
            f"[TEST]  MAE={test_m['mae']:.4f}  RMSE={test_m['rmse']:.4f}  "
            f"MAPE={test_m['mape']:.2f}%  WAPE={test_m['wape']:.2f}%  MSE={test_m['mse']:.4f}"
        )
        logger.info("=" * 70)

main()
